<a href="https://www.kaggle.com/code/zn1350/chess-game-training?scriptVersionId=334041653" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

<a href="https://www.kaggle.com/code/zn1350/chess-game-training?scriptVersionId=334019623" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# 在 Kaggle GPU 上训练小型评估网络

本 notebook 用于在 Kaggle Kernel（GPU）上做最小量的离线训练：
- 使用当前项目的 `ChessGame` 与 `AIEngine._evaluate` 作为标签来源（快速生成训练样本）
- 训练一个小型 CNN 估值网络（board -> scalar）
- 保存训练好的权重到 `model.pth`，供线上推理使用。

说明：这是轻量演示性脚本，目标是尽可能少的训练时间内得到可用权重。

In [1]:
# cell1
import os, sys, random, time
# 若 Kaggle 环境缺少 torch，可取消下面注释安装（通常 Kaggle 已预装）
# !pip install -q torch torchvision --extra-index-url https://download.pytorch.org/whl/cu118

# 明确追加Kaggle工作目录到Python路径，确保能找到自定义模块
WORK_DIR = '/kaggle/input/models/zn1350/chess-game/pytorch/default/1'
sys.path.append(WORK_DIR)
# 注意：不再切换工作目录，避免路径混乱，直接使用绝对路径导入
# os.chdir(WORK_DIR)

# 导入项目中的棋盘与评估函数（确保chess_game.py、ai_engine.py在WORK_DIR下）
from chess_game import (
    ChessGame, R_KING, R_GUARD, R_BISHOP, R_ROOK, R_KNIGHT, R_CANNON, R_PAWN,
    B_KING, B_GUARD, B_BISHOP, B_ROOK, B_KNIGHT, B_CANNON, B_PAWN, EMPTY
)
from ai_engine import AIEngine

In [2]:
# cell2
import os, sys
# 追加当前工作目录到路径（Kaggle Notebook 运行时工作目录为 /kaggle/working）
sys.path.append('.')
print("当前工作目录:", os.getcwd())

当前工作目录: /kaggle/working


In [3]:
# cell3
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split
import copy

In [4]:
# cell4
# 棋盘编码：15 通道 = 14 棋子通道 + 1 行棋方通道（全1表示红方走，全0表示黑方走）
PIECE_LIST = [
    R_KING, R_GUARD, R_BISHOP, R_ROOK, R_KNIGHT, R_CANNON, R_PAWN,
    B_KING, B_GUARD, B_BISHOP, B_ROOK, B_KNIGHT, B_CANNON, B_PAWN,
]
PIECE_TO_IDX = {p: i for i, p in enumerate(PIECE_LIST)}

def board_to_tensor(game: ChessGame):
    """
    将局面转换为 15x10x9 的张量，额外通道指示当前行棋方。
    """
    board = game.board
    x = np.zeros((15, 10, 9), dtype=np.float32)
    for r in range(10):
        for c in range(9):
            p = board[r][c]
            if p != EMPTY:
                idx = PIECE_TO_IDX.get(p)
                if idx is not None:
                    x[idx, r, c] = 1.0
    # 行棋方通道：假设 ChessGame.current_turn 为 'red' 或 'black'
    if game.current_turn == 'red':   # 根据实际实现调整，可能是字符串或整数
        x[14, :, :] = 1.0
    return x

In [5]:
# cell5
def generate_dataset(n_positions=5000, max_rand_moves=8, seed=42):
    """
    生成训练数据，返回归一化后的 X, y_norm 以及用于反归一化的 (y_mean, y_std)。
    局面数量增加至 5000，可自行调整。
    """
    random.seed(seed)
    np.random.seed(seed)
    ai = AIEngine()
    X = []
    y = []
    for i in range(n_positions):
        g = ChessGame()
        moves = random.randint(0, max_rand_moves)
        for _ in range(moves):
            allm = g.get_all_moves(g.current_turn)
            if not allm:
                break
            mv = random.choice(allm)
            g.make_move(*mv)
        X.append(board_to_tensor(g))
        val = ai._evaluate(g)
        y.append(float(val))
    X = np.stack(X, axis=0)
    y = np.array(y, dtype=np.float32)
    # 标签归一化
    y_mean = y.mean()
    y_std = y.std()
    y_norm = (y - y_mean) / (y_std + 1e-8)
    return X, y_norm, y_mean, y_std

# 生成数据（可酌情调整 n_positions）
X, y_norm, y_mean, y_std = generate_dataset(n_positions=5000, max_rand_moves=10, seed=42)
print('X shape:', X.shape, 'y_norm shape:', y_norm.shape)
print('Original y mean:', y_mean, 'std:', y_std)

X shape: (5000, 15, 10, 9) y_norm shape: (5000,)
Original y mean: -128.0652 std: 178.16862


In [6]:
# cell6
class BoardDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y).unsqueeze(1)   # shape: [N,1]
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# 划分训练/验证集 (80% / 20%)
dataset = BoardDataset(X, y_norm)
n_total = len(dataset)
n_train = int(0.8 * n_total)
n_val = n_total - n_train
train_dataset, val_dataset = random_split(dataset, [n_train, n_val])
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
print(f'Train samples: {n_train}, Val samples: {n_val}')

Train samples: 4000, Val samples: 1000


In [7]:
# cell7
# 更深的残差估值网络
class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch, ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(ch)
        )
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.relu(x + self.conv(x))

class EvalNet(nn.Module):
    def __init__(self, in_ch=15):
        super().__init__()
        self.tower = nn.Sequential(
            nn.Conv2d(in_ch, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            ResBlock(64),
            ResBlock(64),
            nn.Conv2d(64, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            ResBlock(128),
            ResBlock(128),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.tower(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = EvalNet().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()
# 学习率衰减
scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=15, gamma=0.5)
print(f'Using device: {device}')

Using device: cuda


In [8]:
# cell8
epochs = 60   # 增大训练轮次
best_val_loss = float('inf')
best_model_state = None

for ep in range(epochs):
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        out = model(xb)
        loss = loss_fn(out, yb)
        loss.backward()
        opt.step()
        train_loss += loss.item() * xb.size(0)
    train_loss /= len(train_loader.dataset)

    # 验证
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            loss = loss_fn(out, yb)
            val_loss += loss.item() * xb.size(0)
    val_loss /= len(val_loader.dataset)

    scheduler.step()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())

    print(f'Epoch {ep+1:02d}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | LR: {opt.param_groups[0]["lr"]:.6f}')

# 加载最佳验证模型
model.load_state_dict(best_model_state)
print(f'Best Val Loss: {best_val_loss:.4f}')

Epoch 01/60 | Train Loss: 0.2806 | Val Loss: 0.2711 | LR: 0.001000
Epoch 02/60 | Train Loss: 0.1001 | Val Loss: 0.3994 | LR: 0.001000
Epoch 03/60 | Train Loss: 0.0863 | Val Loss: 0.1649 | LR: 0.001000
Epoch 04/60 | Train Loss: 0.0715 | Val Loss: 0.2443 | LR: 0.001000
Epoch 05/60 | Train Loss: 0.0693 | Val Loss: 0.7536 | LR: 0.001000
Epoch 06/60 | Train Loss: 0.0599 | Val Loss: 0.1713 | LR: 0.001000
Epoch 07/60 | Train Loss: 0.0732 | Val Loss: 0.2794 | LR: 0.001000
Epoch 08/60 | Train Loss: 0.0565 | Val Loss: 0.0977 | LR: 0.001000
Epoch 09/60 | Train Loss: 0.0462 | Val Loss: 0.5435 | LR: 0.001000
Epoch 10/60 | Train Loss: 0.0516 | Val Loss: 0.0958 | LR: 0.001000
Epoch 11/60 | Train Loss: 0.0600 | Val Loss: 0.0825 | LR: 0.001000
Epoch 12/60 | Train Loss: 0.0420 | Val Loss: 0.0850 | LR: 0.001000
Epoch 13/60 | Train Loss: 0.0392 | Val Loss: 0.0790 | LR: 0.001000
Epoch 14/60 | Train Loss: 0.0323 | Val Loss: 0.1402 | LR: 0.001000
Epoch 15/60 | Train Loss: 0.0422 | Val Loss: 0.1132 | LR: 0.00

In [9]:
# cell9
os.makedirs('/kaggle/working', exist_ok=True)
save_dict = {
    'state_dict': model.state_dict(),
    'y_mean': y_mean,      # 保存归一化参数，推理时用于反变换
    'y_std': y_std,
    'in_channels': 15,     # 便于加载时确认
}
save_path = '/kaggle/working/model_improved.pth'
torch.save(save_dict, save_path)
print('Model and normalization parameters saved to', save_path)

Model and normalization parameters saved to /kaggle/working/model_improved.pth


## 接下来在项目中使用训练好的模型
在 `ai_engine.py` 的 `_evaluate` 中加载 `model.pth` 并用 `model.forward(board_tensor)` 得到估值（替换或与原评估混合）。建议先在 CPU 上做推理速度测试，再部署到实时系统。